# ML Assignment 2 - Online Shoppers Purchase-Intent Classifiers

End-to-end run of the assignment on the UCI **Online Shoppers Purchasing Intention** dataset
(12,330 sessions, 17 features, target `Revenue`):

1. clean the data and make a stratified 80/20 split,
2. tune and train Logistic Regression, Decision Tree, KNN, Naive Bayes and SVM,
3. evaluate every model with Accuracy, AUC, Precision, Recall, F1 and MCC on the hold-out split,
4. save the fitted pipelines (`.joblib`) and the hold-out file (`test_data.csv`).

All logic lives in `train_models.py` and `shopper_utils.py`, which the Streamlit app (`app.py`) also uses.
Run the cells from the repository root (the folder that contains `train_models.py`).
If a package is missing on the machine, run `pip install -r requirements.txt` in a terminal
(on an older Python, `pip install scikit-learn pandas numpy matplotlib seaborn joblib` also works).

In [ ]:
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import ConfusionMatrixDisplay

import shopper_utils as su
import train_models as tm

print("Python", platform.python_version(), "| scikit-learn", sklearn.__version__, "| pandas", pd.__version__)

## 1. Train and evaluate

`OUT_DIR` keeps this run separate from the committed `model/` folder.
`TUNE = True` repeats the full grid search (about 2-3 minutes on a 16-core laptop, longer on a small machine);
`TUNE = False` trains with fixed hyper-parameters and is much faster.

In [ ]:
TUNE = True
OUT_DIR = "lab_run"

metrics = tm.run(out_dir=OUT_DIR, tune=TUNE)

## 2. Comparison table (hold-out test split)

In [ ]:
metrics.set_index("ML Model Name").round(4)

## 3. Confusion matrices from the *saved* models

The models are reloaded from disk and score `test_data.csv` without any re-training.

In [ ]:
models = su.load_models(Path(OUT_DIR) / "model")
test = pd.read_csv(Path(OUT_DIR) / "test_data.csv")
X_test, y_test = su.prepare_features(test), su.prepare_target(test)

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, (name, model) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, model.predict(X_test), display_labels=su.CLASS_NAMES, cmap="GnBu", colorbar=False, ax=ax
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()

check = su.metrics_table(su.evaluate_all(models, X_test, y_test))
print("Reloaded models reproduce the table above:", np.allclose(check[su.METRICS], metrics[su.METRICS]))

## 4. Launch the app

```
streamlit run app.py
```

Upload `test_data.csv` (or any CSV with the same columns) in the sidebar to evaluate every saved model.